# zh-en-subtitle-agent: run in Colab with a local model (Ollama)

Write code in Codespaces → `git push` → run here on a GPU.

**One-time setup (🔑 Secrets panel on the left):**
- `GH_TOKEN`: a GitHub fine-grained token with **read-only Contents** access to this repo only (needed while the repo is private)
- `LANGSMITH_API_KEY` *(optional)*: turns on tracing so you can see every tool call

Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the latest code

In [ ]:
import os
from google.colab import userdata

REPO = "rgbprocessing/zh-en-subtitle-agent"
DIR = "/content/zh-en-subtitle-agent"

if not os.path.exists(DIR):
    token = userdata.get("GH_TOKEN")
    !git clone -q https://{token}@github.com/{REPO}.git {DIR}
else:
    !git -C {DIR} pull -q
%cd {DIR}
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Start Ollama and download the model
The model download (~5 GB) happens once per session.

In [ ]:
import subprocess, time, urllib.request

!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(30):                      # wait until the server answers
    try:
        urllib.request.urlopen("http://localhost:11434"); break
    except Exception:
        time.sleep(1)

MODEL = "qwen2.5:7b"     # strong at Chinese, supports tool calling. Bigger GPU? try "qwen2.5:14b"
!ollama pull {MODEL}

## 3. (Optional) LangSmith tracing

In [ ]:
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "zh-en-subtitle-agent"
    print("Tracing on → smith.langchain.com")
except Exception:
    print("No LANGSMITH_API_KEY secret; running without tracing.")

## 4. Tests + dry run (no LLM)

In [ ]:
!python -m unittest discover -s tests
!python -m subtitle_agent.run sample/manifest.toml --dry-run --out out/ --show-pending 0

## 5. Full agent run on the sample

In [ ]:
!python -m subtitle_agent.run sample/manifest.toml --model ollama:{MODEL} --out out/

In [ ]:
print(open("out/demo_en.srt", encoding="utf-8").read())
print(open("out/demo_notes.md", encoding="utf-8").read())
!python -m subtitle_agent.eval_cer --ref sample/reference_zh.srt --hyp out/demo_zh.srt --manifest sample/manifest.toml

## 6. Real episodes (from Google Drive)

Keep video, audio, and OCR/ASR outputs in Drive, **not** in the repo. Put a `manifest.toml` in each episode folder with paths relative to it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

EP = "/content/drive/MyDrive/dream_team/ep05"     # ← change to your folder
!python -m subtitle_agent.run {EP}/manifest.toml --dry-run --out {EP}/agent_out

In [ ]:
# when the dry run looks right:
!python -m subtitle_agent.run {EP}/manifest.toml --model ollama:{MODEL} --out {EP}/agent_out